# Guided Project 4: AI Job Interview Coach

## Business Scenario

In the modern job market, candidates often face challenges in preparing for interviews in a structured, confident, and professional manner. HR teams and career counselors seek an AI-powered Interview Coach that can simulate realistic interview scenarios, evaluate candidate responses, and provide constructive feedback.

The goal is to build a **Job Interview Coaching Assistant** using LangChain that leverages Prompt Templates and Few-Shot Examples. By exposing the AI to examples of strong and weak responses, the assistant can guide learners toward delivering structured, impactful, and professional answers.



## Problem Statement

Design and implement an AI Interview Coach using LangChain that:

* Utilizes a **Prompt Template** to enforce structured responses (e.g., introduction, main answer, feedback).
* Incorporates **Few-Shot Examples** to demonstrate what constitutes strong versus weak interview answers.
* Simulates **realistic interview questions** across multiple domains (e.g., software engineering, data science, business analysis).
* Provides **personalized, step-by-step feedback** to help candidates improve their performance.



## Step-by-Step Approach

**Step 1: Load an LLM**

* Use AWS Bedrock integrated LLM via LangChain.

**Step 2: Define a Prompt Template**

* Create a template with placeholders such as:

  * `{job_role}` → target role for the interview
  * `{interview_question}` → question asked by the interviewer
  * `{candidate_response}` → candidate’s answer to be evaluated

**Step 3: Add Few-Shot Examples**

* Provide 2–3 examples of interview questions with both weak and strong answers.
* Demonstrate how structured feedback should be presented.

**Step 4: Build the Chain**

* Use `FewShotPromptTemplate` in LangChain to combine examples with the dynamic template.

**Step 5: Simulate Interview Q\&A**

* User provides a job role and candidate response.
* The system:

  1. Reframes the candidate’s answer.
  2. Compares it against strong examples.
  3. Provides structured feedback, including strengths, areas for improvement, and final recommendations.

**Step 6: Test with Sample Questions**
Example questions:

* *“Tell me about yourself.”*
* *“Why do you want to join our company?”*
* *“Explain a challenging project you worked on.”*

**Step 7: Evaluation & Enhancement**

* Ensure the system maintains a professional, coaching-oriented tone.
* Test across different roles (e.g., Software Engineer, Data Analyst, Business Analyst).
* Extend to multi-turn sessions where the AI recalls prior candidate responses.



In [1]:
# Capstone Project: AI Job Interview Coach
# Using LangChain + Hugging Face (No OpenAI)

# Step 1: Imports
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser



In [2]:
from langchain_aws import ChatBedrockConverse
llm=ChatBedrockConverse(model='cohere.command-r-plus-v1:0', #amazon.nova-lite-v1:0
                       aws_access_key_id='',
                       aws_secret_access_key='',
                       region_name='us-east-1',max_tokens=200)
llm.invoke("Hi").content

'Hello! How can I help you today?'

In [3]:

# Step 2: Define few-shot examples
examples = [
    {
        "interview_question": "Tell me about yourself.",
        "candidate_response": "I am John. I like coding. I need a job.",
        "feedback": "Weak answer. It lacks structure and doesn't highlight professional achievements. Instead, start with your background, key skills, and end with how it aligns with the job."
    },
    {
        "interview_question": "Tell me about yourself.",
        "candidate_response": "I am John, a software engineer with 5 years of experience in backend development using Python and cloud technologies. I enjoy solving real-world problems with scalable systems, and I am excited to apply these skills to contribute to your company’s innovative projects.",
        "feedback": "Strong answer. It is structured, professional, and connects personal skills with the role."
    },
    {
        "interview_question": "Why do you want to join our company?",
        "candidate_response": "Because you pay well.",
        "feedback": "Weak answer. Focus instead on the company’s values, culture, or projects that resonate with you."
    },
    {
        "interview_question": "Why do you want to join our company?",
        "candidate_response": "I admire your company’s focus on innovation in AI and data-driven solutions. I believe my background in machine learning aligns with your mission, and I’m eager to contribute to impactful projects that improve customer experiences.",
        "feedback": "Strong answer. It highlights motivation, alignment with the company’s vision, and relevant skills."
    }
]

# Step 3: Define example and overall template
example_template = """
Interview Question: {interview_question}
Candidate Response: {candidate_response}
Feedback: {feedback}
"""

example_prompt = PromptTemplate(
    input_variables=["interview_question", "candidate_response", "feedback"],
    template=example_template,
)

overall_template = """
You are an AI Interview Coach.
Evaluate the candidate's answer and provide structured feedback.

Follow this format:
1. Strengths
2. Improvements
3. Final Recommendation (Strong / Weak)

Examples: {{examples}}
Interview Question: {interview_question}
Candidate Response: {candidate_response}
Feedback:
"""

# Step 4: Create Few-Shot Prompt Template
prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix=overall_template,
    input_variables=["interview_question", "candidate_response"],
)

# Step 5: Create LLM Chain

chain = prompt | llm | StrOutputParser()


In [4]:
from rich import print
print(chain )

RunnableSequence(
    first=FewShotPromptTemplate(
        input_variables=['candidate_response', 'interview_question'],
        input_types={},
        partial_variables={},
        examples=[
            {
                'interview_question': 'Tell me about yourself.',
                'candidate_response': 'I am John. I like coding. I need a job.',
                'feedback': "Weak answer. It lacks structure and doesn't highlight professional achievements. 
Instead, start with your background, key skills, and end with how it aligns with the job."
            },
            {
                'interview_question': 'Tell me about yourself.',
                'candidate_response': 'I am John, a software engineer with 5 years of experience in backend 
development using Python and cloud technologies. I enjoy solving real-world problems with scalable systems, and I 
am excited to apply these skills to contribute to your company’s innovative projects.',
                'feedback': 'Strong answer. It is structured, professional, and connects personal skills with the 
role.'
            },
            {
                'interview_question': 'Why do you want to join our company?',
                'candidate_response': 'Because you pay well.',
                'feedback': 'Weak answer. Focus instead on the company’s values, culture, or projects that resonate
with you.'
            },
            {
                'interview_question': 'Why do you want to join our company?',
                'candidate_response': 'I admire your company’s focus on innovation in AI and data-driven solutions.
I believe my background in machine learning aligns with your mission, and I’m eager to contribute to impactful 
projects that improve customer experiences.',
                'feedback': 'Strong answer. It highlights motivation, alignment with the company’s vision, and 
relevant skills.'
            }
        ],
        example_prompt=PromptTemplate(
            input_variables=['candidate_response', 'feedback', 'interview_question'],
            input_types={},
            partial_variables={},
            template='\nInterview Question: {interview_question}\nCandidate Response: 
{candidate_response}\nFeedback: {feedback}\n'
        ),
        suffix="\nYou are an AI Interview Coach.\nEvaluate the candidate's answer and provide structured 
feedback.\n\nFollow this format:\n1. Strengths\n2. Improvements\n3. Final Recommendation (Strong / 
Weak)\n\nExamples: {{examples}}\nInterview Question: {interview_question}\nCandidate Response: 
{candidate_response}\nFeedback:\n"
    ),
    middle=[
        ChatBedrockConverse(
            profile={
                'max_input_tokens': 128000,
                'max_output_tokens': 4096,
                'image_inputs': False,
                'audio_inputs': False,
                'video_inputs': False,
                'image_outputs': False,
                'audio_outputs': False,
                'video_outputs': False,
                'reasoning_output': False,
                'tool_calling': True,
                'image_url_inputs': True,
                'pdf_inputs': True,
                'pdf_tool_message': True,
                'image_tool_message': True
            },
            client=<botocore.client.BedrockRuntime object at 0x7cd5e1f855d0>,
            bedrock_client=<botocore.client.Bedrock object at 0x7cd5e14e55a0>,
            model_id='cohere.command-r-plus-v1:0',
            max_tokens=200,
            region_name='us-east-1',
            aws_access_key_id=SecretStr(''),
            aws_secret_access_key=SecretStr(''),
            provider='cohere',
            supports_tool_choice_values=()
        )
    ],
    last=StrOutputParser()
)

In [5]:
response = chain.invoke({
    "interview_question": "Explain a challenging project you worked on.",
    "candidate_response": "I led a project migrating legacy databases to cloud, improving data processing speed by 40%. I faced performance issues initially but resolved them with query optimization and parallelization."
})

print(response)

Sure! Here is some structured feedback for the candidate's answer:

Interview Question: Explain a challenging project you worked on.
Candidate Response: I led an ambitious project to migrate legacy databases to the cloud, which resulted in a 
significant improvement in data processing speed, achieving a 40% increase in efficiency. While we encountered 
performance issues at the outset, my team and I successfully overcame these challenges through strategic query 
optimization and the implementation of parallel processing techniques.

Feedback:

1. Strengths:
   - Leadership: The candidate took initiative and led a project, demonstrating leadership skills and the ability 
to drive change.
   - Problem Solving: They identified and resolved performance issues, showcasing analytical and problem-solving 
capabilities.
   - Technical Skills: The mention of "query optimization" and "parallelization" indicates a strong understanding 
of technical concepts and their practical application.
   - Result-Oriented: The project resulted in a measurable improvement of 40% in data processing speed,